# Notebook 09: Color Waypoint Challenge

## ADAS Connection
Real autonomous vehicles do not rely on a single system to make decisions. They layer
computer vision, sensor data, and AI reasoning together. In this challenge you will
see what happens when you swap those layers in and out -- and which one performs better
under pressure.

Your robot will respond to colored cards held up by the judge. Each color maps to a
driving action. You will run the challenge twice -- once using pure computer vision,
once using an AI model as the decision maker.

---

## The Rules

| Card Color | Action |
|------------|--------|
| Red | Stop for 2 seconds |
| Green | Go forward |
| Blue | Turn left |
| Yellow | Turn right |
| Orange | Mission complete -- stop and celebrate |

- The judge holds up a card and your robot must respond correctly
- Decisions print to screen so the judges can follow your robot's reasoning
- In **CV Mode** your robot decides instantly using color detection
- In **AI Mode** your robot asks Phi-3 Mini what to do and acts on the answer

---

## Before You Start

> **Step 1:** Make sure your robot is connected to the school WiFi
>
> **Step 2:** Update the AI server IP address in the configuration cell below -- your
> instructor will give you the address. If it is wrong, AI Mode will not work.
>
> **Step 3:** Run the setup cell, then the configuration cell, then the challenge cell.

---

## Setup
Run this cell once to initialize the camera, motors, and AI connection.

In [ ]:
import cv2
import numpy as np
import RPi.GPIO as GPIO
import requests
import json
import time
import ipywidgets as widgets
from IPython.display import display

# ── Camera ────────────────────────────────────────────────────
if 'cap' not in dir() or not cap.isOpened():
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M','J','P','G'))
    for _ in range(20):
        cap.read()
    print('Camera initialized!')
else:
    print('Camera already open -- skipping init.')

# ── Motor pins (BCM numbering) ────────────────────────────────
LEFT_GO   = 20
LEFT_BACK = 21
LEFT_PWM  = 16
RIGHT_GO  = 19
RIGHT_BACK = 6
RIGHT_PWM  = 13

GPIO.setmode(GPIO.BCM)
GPIO.setwarnings(False)
for pin in [LEFT_GO, LEFT_BACK, LEFT_PWM, RIGHT_GO, RIGHT_BACK, RIGHT_PWM]:
    GPIO.setup(pin, GPIO.OUT)

pwm_left  = GPIO.PWM(LEFT_PWM,  100)
pwm_right = GPIO.PWM(RIGHT_PWM, 100)
pwm_left.start(0)
pwm_right.start(0)

# ── Motor functions ───────────────────────────────────────────
def forward(speed=60):
    GPIO.output(LEFT_GO,   GPIO.HIGH)
    GPIO.output(LEFT_BACK, GPIO.LOW)
    GPIO.output(RIGHT_GO,  GPIO.HIGH)
    GPIO.output(RIGHT_BACK,GPIO.LOW)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)

def stop():
    GPIO.output(LEFT_GO,   GPIO.LOW)
    GPIO.output(LEFT_BACK, GPIO.LOW)
    GPIO.output(RIGHT_GO,  GPIO.LOW)
    GPIO.output(RIGHT_BACK,GPIO.LOW)
    pwm_left.ChangeDutyCycle(0)
    pwm_right.ChangeDutyCycle(0)

def turn_left(speed=60, duration=0.5):
    GPIO.output(LEFT_GO,   GPIO.LOW)
    GPIO.output(LEFT_BACK, GPIO.HIGH)
    GPIO.output(RIGHT_GO,  GPIO.HIGH)
    GPIO.output(RIGHT_BACK,GPIO.LOW)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)
    time.sleep(duration)
    stop()

def turn_right(speed=60, duration=0.5):
    GPIO.output(LEFT_GO,   GPIO.HIGH)
    GPIO.output(LEFT_BACK, GPIO.LOW)
    GPIO.output(RIGHT_GO,  GPIO.LOW)
    GPIO.output(RIGHT_BACK,GPIO.HIGH)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)
    time.sleep(duration)
    stop()

# ── Color ranges (HSV) ────────────────────────────────────────
COLOR_RANGES = {
    'red':    ([0,   43,  46],  [10,  255, 255]),
    'green':  ([35,  43,  46],  [77,  255, 255]),
    'blue':   ([100, 43,  46],  [124, 255, 255]),
    'yellow': ([26,  43,  46],  [34,  255, 255]),
    'orange': ([11,  43,  46],  [25,  255, 255]),
}

# ── Helper functions ──────────────────────────────────────────
def bgr8_to_jpeg(frame):
    return bytes(cv2.imencode('.jpg', frame)[1])

def detect_color(frame):
    """Return the dominant color name detected in the frame, or None."""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    best_color = None
    best_count = 500   # minimum pixel threshold to count as a detection
    for color, (lower, upper) in COLOR_RANGES.items():
        mask = cv2.inRange(hsv, np.array(lower), np.array(upper))
        count = cv2.countNonZero(mask)
        if count > best_count:
            best_count = count
            best_color = color
    return best_color

def ask_ai(color, server_ip, server_port):
    """Ask Phi-3 what action to take for a given color."""
    prompt = (
        f'You are the decision-making system for an autonomous robot. '
        f'The robot\'s camera has detected the color {color}. '
        f'Based on the following rules, respond with ONLY the action word and nothing else.\n'
        f'Rules:\n'
        f'red = STOP\n'
        f'green = GO\n'
        f'blue = LEFT\n'
        f'yellow = RIGHT\n'
        f'orange = DONE\n'
        f'What is the action for {color}?'
    )
    try:
        response = requests.post(
            f'http://{server_ip}:{server_port}/api/generate',
            json={'model': 'phi3:mini', 'prompt': prompt, 'stream': False},
            timeout=5
        )
        result = response.json()['response'].strip().upper()
        # extract just the action word in case the model adds extra text
        for action in ['STOP', 'GO', 'LEFT', 'RIGHT', 'DONE']:
            if action in result:
                return action
        return None
    except Exception as e:
        print(f'  AI error: {e}')
        return None

def execute_action(action):
    """Execute the driving action."""
    if action == 'GO':
        forward(DRIVE_SPEED)
    elif action == 'STOP':
        stop()
        time.sleep(2)
        forward(DRIVE_SPEED)
    elif action == 'LEFT':
        turn_left(DRIVE_SPEED, TURN_DURATION)
        forward(DRIVE_SPEED)
    elif action == 'RIGHT':
        turn_right(DRIVE_SPEED, TURN_DURATION)
        forward(DRIVE_SPEED)
    elif action == 'DONE':
        stop()

print('Setup complete!')
print('Motors, camera, and color detection ready.')
print()
print('Next: run the configuration cell.')

---

## Configuration
**Run this cell to set your mode and server address before the challenge.**

Switch between `CV` and `AI` mode here and re-run to change modes between runs.

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
MODE          = 'CV'              # 'CV' for computer vision, 'AI' for Phi-3
SERVER_IP     = 'YOUR.IP.HERE'    # ask your instructor for the AI server IP
SERVER_PORT   = 11434             # default Ollama port -- do not change
DRIVE_SPEED   = 60                # motor speed (0-100)
TURN_DURATION = 0.5               # seconds per turn -- tune this for your kit
SCAN_INTERVAL = 0.2               # seconds between camera scans
# ═══════════════════════════════════════

print(f'Mode:         {MODE}')
print(f'Drive speed:  {DRIVE_SPEED}')
print(f'Turn duration:{TURN_DURATION}s')
if MODE == 'AI':
    print(f'AI server:    http://{SERVER_IP}:{SERVER_PORT}')
    if SERVER_IP == 'YOUR.IP.HERE':
        print()
        print('  WARNING: You have not set the server IP address!')
        print('  AI Mode will not work until you update SERVER_IP.')
print()
print('Configuration ready. Run the challenge cell when the judge says go!')

---

## Challenge
Run this cell to start. The robot will scan for color cards and respond.
Run the **Stop** cell at any time to end the run.

In [ ]:
print('=' * 50)
print(f'  CHALLENGE STARTING -- Mode: {MODE}')
print('=' * 50)
print()

challenge_running = True
last_action = None

# display widget for live camera feed
feed_widget = widgets.Image(format='jpeg', width=640, height=480)
display(feed_widget)

forward(DRIVE_SPEED)
print('Robot moving -- show a color card to the camera!')
print()

while challenge_running:
    # flush buffer and grab fresh frame
    for _ in range(5):
        cap.read()
    ret, frame = cap.read()
    if not ret:
        print('ERROR: Camera read failed.')
        break

    # detect color
    color = detect_color(frame)

    if color:
        if MODE == 'CV':
            # CV Mode -- map color to action directly
            action_map = {
                'red':    'STOP',
                'green':  'GO',
                'blue':   'LEFT',
                'yellow': 'RIGHT',
                'orange': 'DONE',
            }
            action = action_map.get(color)
            print(f'[CV]  Detected: {color:8s}  -->  Action: {action}')

        elif MODE == 'AI':
            # AI Mode -- ask Phi-3 what to do
            print(f'[AI]  Detected: {color:8s}  -->  Asking Phi-3...')
            action = ask_ai(color, SERVER_IP, SERVER_PORT)
            if action:
                print(f'[AI]  Phi-3 says: {action}')
            else:
                print(f'[AI]  No valid action returned -- continuing forward.')

        # execute action if it changed
        if action and action != last_action:
            execute_action(action)
            last_action = action
            if action == 'DONE':
                print()
                print('=' * 50)
                print('  MISSION COMPLETE!')
                print('=' * 50)
                challenge_running = False

    # annotate and display frame
    label = f'Mode: {MODE}  |  Detected: {color if color else "none"}'
    cv2.putText(frame, label, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    feed_widget.value = bgr8_to_jpeg(frame)

    time.sleep(SCAN_INTERVAL)

---
## Stop
Run this cell at any time to stop the robot.

In [ ]:
challenge_running = False
stop()
print('Robot stopped.')

---

## Debrief

Talk through these questions with your team:

1. Which mode was faster to respond -- CV or AI? Why?
2. Which mode was more reliable? Did either mode make mistakes?
3. In a real self-driving car, when would you trust AI reasoning over direct computer vision?
4. What would you change about your color ranges or motor tuning to improve performance?

---

## Always clean up when you are done!

In [ ]:
stop()
cap.release()
pwm_left.stop()
pwm_right.stop()
GPIO.cleanup()
print('All cleaned up.')